# Lab 05: Restartable optical batch

This notebook covers the six requests and the recovery cases using the supplied Sentinel-2 fixtures. The requests use three dates and two AOIs near Kraków. Cloud suitability is not assessed.


## 1. Open the package and inspect the evidence

Run the notebook from the Lab05 directory. Keep the supplied Python files beside it.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, sqlite3, subprocess, sys
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import batch_plan, batch_repairs, processing_store

ROOT = Path.cwd().resolve()
assert (ROOT / 'lab05_cli.py').is_file(), 'Open this notebook from lab05_solution.'
EVIDENCE = ROOT / 'evidence'
EVIDENCE.mkdir(exist_ok=True)

def cli(*args, expected=(0,)):
    child = subprocess.run([sys.executable, str(ROOT / 'lab05_cli.py'), *map(str, args)], capture_output=True, text=True)
    try:
        response = json.loads(child.stdout) if child.stdout.strip() else None
    except json.JSONDecodeError:
        response = {'stdout': child.stdout}
    with (EVIDENCE / 'commands.jsonl').open('a') as stream:
        stream.write(json.dumps({'recorded_at': processing_store.now(), 'args': list(map(str, args)),
                                'exit_code': child.returncode, 'response': response,
                                'stderr': child.stderr}) + '\n')
    assert child.returncode in expected, (child.returncode, child.stdout[-2500:], child.stderr[-2500:])
    if child.returncode == 87:
        return {'exit_code': 87, 'note': 'The isolated coordinator stopped abruptly.'}
    return response

def check(group='all'):
    child = subprocess.run([sys.executable, str(ROOT / 'check_lab05.py'), group], capture_output=True, text=True)
    print(child.stdout)
    return child.returncode == 0

def query(manifest_path, sql, params=()):
    manifest = batch_plan.load(manifest_path)
    with sqlite3.connect(manifest['_database']) as connection:
        connection.row_factory = sqlite3.Row
        connection.execute('PRAGMA foreign_keys=ON')
        return [dict(row) for row in connection.execute(sql, params)]

SQL = []
statement = ''
for line in (ROOT / 'investigation_queries.sql').read_text().splitlines(True):
    statement += line
    if sqlite3.complete_statement(statement):
        SQL.append(statement)
        statement = ''
assert len(SQL) == 6

def investigate(manifest_path, number, **params):
    manifest = batch_plan.load(manifest_path)
    return query(manifest_path, SQL[number - 1], {'batch_id': manifest['batch_id'], **params})

LATEST = """SELECT m.member_id, o.observation_id, o.state, o.reason, o.observed_at, o.evidence_json
FROM batch_members m LEFT JOIN batch_observations o
ON o.batch_id=m.batch_id AND o.member_id=m.member_id
AND o.observation_id=(SELECT MAX(x.observation_id) FROM batch_observations x
                     WHERE x.batch_id=m.batch_id AND x.member_id=m.member_id)
WHERE m.batch_id=:batch_id ORDER BY m.member_id"""

def capture(name, manifest_path):
    path = EVIDENCE / (name + '.json')
    if path.exists():
        return json.loads(path.read_text())
    manifest = batch_plan.load(manifest_path)
    latest = query(manifest_path, LATEST, {'batch_id': manifest['batch_id']})
    summary = batch_repairs.summarize_batch([{**row, 'observation_id': row['observation_id'] if row['observation_id'] is not None else -1} for row in latest])
    value = {'captured_at': processing_store.now(), 'batch_id': manifest['batch_id'],
             'manifest': str(manifest_path), 'latest': latest, 'summary': summary,
             'attempts': query(manifest_path, 'SELECT * FROM processing_attempts ORDER BY started_at, attempt_id'),
             'no_attempts': investigate(manifest_path, 1), 'recovered_requests': investigate(manifest_path, 2),
             'unavailable_products': investigate(manifest_path, 3),
             'missing_roles': investigate(manifest_path, 5), 'running': investigate(manifest_path, 6)}
    path.write_text(json.dumps(value, indent=2))
    return value

def saved_report(manifest_path, name):
    directory = ROOT / 'reports' / name
    if not directory.exists():
        cli('report', '--manifest', manifest_path, '--output', directory)
    return json.loads((directory / 'report.json').read_text())

print('Python', sys.version.split()[0], 'Rasterio', rasterio.__version__)
fixture = json.loads((ROOT / 'fixtures/manifest.json').read_text())
print('Endpoint:', fixture['endpoint'], 'Collection:', fixture['collection'])
print('Requests:', len(fixture['members']))


In [ ]:
first_member = fixture['members'][0]
for role, entry in first_member['bands'].items():
    with rasterio.open(ROOT / 'fixtures' / entry['file']) as source:
        raw = source.read(1, masked=True)
        print(role, first_member['member_id'], source.shape, source.crs, source.scales, source.offsets, entry['sha256'])
        print('DN median:', float(np.ma.median(raw)))
print('Source transfer bytes:', sum(t['network_bytes'] for t in fixture['transfers']))

Name one property a checksum establishes and one scientific question it cannot answer. Explain why six requests here represent three dates rather than six independent acquisitions.

A matching SHA256 shows that the bytes match the recorded checksum, if that checksum is trusted. It cannot tell us whether the image is cloud free.

There are three acquisitions, each used for two AOIs, so there are six requests but only three dates.


## 2. Design the new tables

Read `SCHEMA_CONTRACT.md`. Write all three tables in `student_batch_schema.sql`. The earlier acquisition and processing schemas are completed in `support/`.

Before writing SQL, sketch the links between a batch, its expected members, their requests, attempts and observations. Decide which records must exist when we know what we want but no calculation has started.

A member's planned request ID exists before its optional request foreign key can be filled. Explain why those are separate fields. Then run the schema check.

In [ ]:
assert check('schema')


The batch and its expected members need to exist before any processing starts. Observations can then record what happens to each member.

```text
batch -> member -> planned request
                -> registered request -> attempts -> outputs
                -> observations -> optional attempt
```

`planned_request_id` identifies work even before a request row exists. `request_id` is an optional foreign key that is filled once the request is registered.


## 3. Repair the coordination logic

The functions in `batch_repairs.py` execute, but the supplied versions make incorrect decisions. Read their contracts and repair them.

`classify_member` needs to distinguish blocked inputs, unresolved execution, verified completion and a permitted new attempt. A finished result may remain usable after its inputs move. An unresolved active execution needs attention even when an older result exists. The attempt budget does not reset when the coordinator restarts.

`verify_result` must reject evidence for another request, mismatching bytes, or missing verification. The adapter does the actual catalogue queries, hashing and raster inspection.

`summarize_batch` must count expected members once each using their newest observations. Repeated observations and joined locations do not create more requested work. Keep members that have no observation yet.

Predict which check will expose each supplied defect before editing.

In [ ]:
for group in ('classify', 'verify', 'summary'):
    assert check(group)


I expected the classify check to catch missing inputs marked as failures and the wrong priority for active attempts and verified results. It also checks validation and retry limits.

The verify check catches results accepted with only an artifact ID. The summary check catches duplicate rows and missing unobserved members. I fixed each function and checked these cases.


## 4. Register expected work and run the batch

Finish the schema and function repairs before creating processing evidence. The next cell imports the captured real inputs into a fresh catalogue and freezes membership. It does not make a processing attempt.

Rerunning the notebook keeps existing work. To begin a separate experiment, choose a new workspace name.

In [ ]:
assert check('all')
MANIFEST = ROOT / 'work/batch.json'
if not MANIFEST.exists():
    print(cli('setup', '--directory', ROOT / 'work'))
initial = capture('main_before', MANIFEST)
assert initial['summary']['expected'] == 6
assert len(initial['attempts']) == 0
assert len(initial['no_attempts']) == 6
initial['no_attempts']


Write a query that lists expected members with no application attempt. Run it now. It should include the whole batch. Keep the query, its parameters and your interpretation in `investigation_queries.sql`.

In [ ]:
# Q1 uses planned identity, so NULL registration cannot hide expected work.
print('Before the first run:', initial['no_attempts'])
print('Current catalogue:', investigate(MANIFEST, 1))


In [ ]:
database = batch_plan.load(MANIFEST)['_database']
before_plan = processing_store.file_hash(database)
plan = cli('plan', '--manifest', MANIFEST)
assert not plan['writes_database']
assert processing_store.file_hash(database) == before_plan
print([(m['member_id'], m['state'], m['reason']) for m in plan['members']])
result = cli('run', '--manifest', MANIFEST)
assert result['complete']
print('Complete:', result['complete'], 'New invocations:', len(result['invocations']))


Predict what a second invocation will change. Will it create another scientific product, another application attempt, or only new batch observations?

In [ ]:
attempts_before = query(MANIFEST, 'SELECT COUNT(*) AS n FROM processing_attempts')[0]['n']
repeat = cli('run', '--manifest', MANIFEST)
attempts_after = query(MANIFEST, 'SELECT COUNT(*) AS n FROM processing_attempts')[0]['n']
print('New invocations:', len(repeat['invocations']))
assert repeat['complete'] and not repeat['invocations']
assert attempts_before == attempts_after

The first run produced six outputs with six attempts. I expected the second run to check and reuse them, adding observations but no new attempts or products. The attempt count stayed at six. The `plan` command also left the database unchanged.


## 5. Look at the results and their report

A contact sheet makes the six results easy to compare. The saved raster values are not clipped. The display uses a fixed interval; blue and magenta identify values outside it.

Export a report, then open its HTML file. Follow one count to its members and inspect a request, producing attempt and verification time. Scientific cloud suitability should remain unassessed.

In [ ]:
report = saved_report(MANIFEST, 'main')
print(report['summary'])
assert report['summary']['counts']['complete'] == 6
assert report['attempt_count'] == 6
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'reports/main/preview.png')))
print('Open:', ROOT / 'reports/main/index.html')
main_after = capture('main_after', MANIFEST)
print('Generated:', report['generated_at'])
print('Observation times:', [m['observed_at'] for m in report['members']])
print('Verification times:', [m['evidence']['result']['checked_at'] for m in report['members']])


If the report says complete, what has it established? Give two reasons that still would not justify calling the image a clear sky vegetation observation. Identify the observation times separately from the report generation time.

Complete means the output bytes, grid and request/attempt tags passed verification at the recorded time. It does not prove the scene is clear or that the pixels represent vegetation: clouds and shadows were not screened, and some pixels may be water or buildings.

`observed_at` is the member observation time; `checked_at` records verification. `generated_at` is when the report was exported.


## 6. Case A · One input disappears

The case helper creates a separate workspace, moves one red input aside, and runs the normal coordinator. Your main workspace and preserved fixtures stay intact.

Predict the expected member count, attempt count and number of completed requests. Then identify the exact blocked member and input role using SQL.

In [ ]:
def ensure_case(name, directory, expected=(0, 1)):
    case_manifest = ROOT / directory / 'batch.json'
    if not case_manifest.exists():
        result = cli('case', '--name', name, '--directory', ROOT / directory, expected=expected)
        print('Created:', name, result.get('manifest', result.get('note')))
    return case_manifest

missing = ensure_case('missing-input', 'cases/missing', expected=(1,))
missing_before = capture('missing_before', missing)
assert missing_before['summary']['expected'] == 6
assert missing_before['summary']['counts']['complete'] == 5
assert missing_before['summary']['counts']['blocked'] == 1
assert len(missing_before['attempts']) == 5
assert len(missing_before['no_attempts']) == 1
saved_report(missing, 'missing_before')
missing_before['missing_roles']


In [ ]:
# Q5 binds the batch ID and reads the missing role from latest adapter evidence.
print('Saved blocked input:', missing_before['missing_roles'])
assert missing_before['missing_roles'][0]['missing_role'] == 'red'
query(missing, """SELECT a.artifact_id, json_extract(a.lineage_json,'$.asset_key') AS role,
 l.status, l.local_path, l.checked_at
FROM processing_artifacts a JOIN processing_artifact_locations l USING(artifact_id)
WHERE a.source_artifact_id IS NOT NULL AND l.status <> 'available'""")


Explain why the blocked member has no failed application attempt. How does expected membership let us see work that never started? Add the query that established your diagnosis to your query file.

I expected six members, five attempts and five completed results. The missing input was the red band for `S2A_T34UDA_20250604T094637_L2A-river`, as Q5 shows.

The input check blocked the request before the application started, so it used no attempt. Q1 starts from the expected members, which keeps unstarted work visible.


In [ ]:
note = json.loads((missing.parent / 'case_note.json').read_text())
if Path(note['held']).exists() and not Path(note['original']).exists():
    print(cli('restore-input', '--manifest', missing))
recovered = cli('run', '--manifest', missing)
assert recovered['complete']
missing_after = capture('missing_after', missing)
assert len(missing_after['attempts']) == 6
assert all(a['status'] == 'succeeded' for a in missing_after['attempts'])
saved_report(missing, 'missing_after')
print('Complete after restoration:', recovered['complete'])
query(missing, 'SELECT status, COUNT(*) AS n FROM processing_attempts GROUP BY status')


## 7. Case B · The worker stops before publication

This helper waits for the actual child process to stop with exit code 86. The child wrote and validated a temporary raster, then exited without running cleanup. Its attempt still says running.

Predict what the coordinator will do with the remaining requests. We still permit only one active processing invocation per catalogue.

In [ ]:
worker = ensure_case('interrupted-worker', 'cases/worker', expected=(1,))
worker_before = capture('worker_before', worker)
assert worker_before['summary']['counts']['unresolved'] == 1
assert worker_before['summary']['counts']['blocked'] == 5
assert len(worker_before['attempts']) == 1
saved_report(worker, 'worker_before')
worker_before['running']


Use SQL and file inspection to distinguish an interrupted execution from a missing input. Record why it is justified to assert that this child stopped. In a real unattended system, a timestamp alone would not establish that.

In [ ]:
running = investigate(worker, 6)
worker_process_evidence = EVIDENCE / 'worker_process_stopped.json'
if running:
    commands = [json.loads(line) for line in (EVIDENCE / 'commands.jsonl').read_text().splitlines()]
    creation = next(c for c in commands if c['args'][:3] == ['case', '--name', 'interrupted-worker'])
    invocation = creation['response']['run']['invocations'][0]
    assert invocation['exit_code'] == 86
    # job_adapter.invoke used subprocess.run, which waited for this actual child.
    evidence = {'recorded_at': processing_store.now(), 'adapter_child_exit_code': 86,
                'wait_evidence': 'case command returned after subprocess.run waited for its worker',
                'attempts': running, 'paths': []}
    for row in running:
        paths = {'attempt_id': row['attempt_id'], 'temporary_exists': Path(row['temp_path']).is_file(),
                 'final_exists': Path(row['final_path']).exists(), 'output_links': row['output_links']}
        assert paths['temporary_exists'] and not paths['final_exists'] and paths['output_links'] == 0
        evidence['paths'].append(paths)
    worker_process_evidence.write_text(json.dumps(evidence, indent=2))
print(json.loads(worker_process_evidence.read_text()))


I expected the other five members to wait while the attempt was still marked running. Q6 showed the attempt at `before_rename`, with a temporary raster and no published output. Unlike the missing-input case, an attempt had started.

The helper waited for the worker to exit with code 86, so I could confirm it had stopped. A timestamp alone would not prove that. I reconciled the attempt and reran the batch; the retry kept the same request ID and got a new attempt ID.


In [ ]:
for row in running:
    assert json.loads(worker_process_evidence.read_text())['adapter_child_exit_code'] == 86
    print(cli('reconcile', '--manifest', worker, '--attempt-id', row['attempt_id'], '--confirm-stopped'))
worker_recovered = cli('run', '--manifest', worker)
assert worker_recovered['complete']
worker_after = capture('worker_after', worker)
assert len(worker_after['attempts']) == 7
assert sum(a['status'] == 'interrupted' for a in worker_after['attempts']) == 1
assert sum(a['status'] == 'succeeded' for a in worker_after['attempts']) == 6
saved_report(worker, 'worker_after')
query(worker, 'SELECT request_id, attempt_id, status, error_code, started_at FROM processing_attempts ORDER BY started_at')


Write a query that finds a request with an interrupted or failed attempt and a later producing attempt. Show both identities. Explain why counting attempts would overstate delivered work.

In [ ]:
recoveries = investigate(worker, 2)
assert len(recoveries) == 1
assert recoveries[0]['earlier_attempt_id'] != recoveries[0]['producing_attempt_id']
print('Recovered request and both attempts:', recoveries)
assert not investigate(worker, 1)
assert not investigate(worker, 3)
cutoff = '2000-01-01T00:00:00Z'
assert not investigate(worker, 4, before=cutoff)
future_cutoff = '2100-01-01T00:00:00Z'
assert len(investigate(worker, 4, before=future_cutoff)) == 6
print('Seven attempts produced six current results; one interrupted attempt produced none.')


Q2 shows the interrupted attempt and the later producing attempt for the same request. There are seven attempts but six results because the interrupted attempt produced no output.


## 8. A decision another person can follow

It is Saturday morning. One member is blocked because an input file is missing. Five other members have verified outputs. Delivery is required on Monday afternoon.

Write a short note stating what is known, what remains uncertain, the next check, and whether you would act now or defer. Explain how your decision changes if an incorrect calibration may have affected already published products.

Save your useful SQL with a question, parameters, empty-result interpretation and next action. Sketch one overview and one detail view using the fields you actually have. Mark any proposed future measurements as unavailable rather than inventing them.

I would investigate on Saturday. Five outputs are verified and one input is missing. I would check the input locations, restore a copy with the matching checksum, rerun the batch and verify the result. The blocked request has used no attempt.

Availability may have changed since the last check, and cloud suitability is unassessed. If calibration may be wrong, I would flag the affected products, pause their use and check the scale and offset. A confirmed correction needs new requests and a new manifest; I would keep the old history.

The overview and detail sketches are in `decision_note.md`.


## Extensions

1. Create `interrupted-coordinator` in a new case directory. The command exits with code 87 after one completed member. Run the same manifest again and prove that only five jobs launch.
2. Create `missing-output`. Inspect both the old succeeded attempt and the new location observations. Recover the output without rewriting history.
3. Add a second copy of a verified output, then remove one copy. Explain why a query using any missing location gives the wrong answer.
4. Repeat the explicit live acquisition command from `README.md` into a new directory. Compare its captured source metadata and subset hashes with the supplied manifest.
5. Change a scientific setting in one configuration and freeze a new manifest. Contrast that new request with a change to an output directory.

Keep extensions separate from your core evidence. Never remove old attempt rows to make a report look clean.

## Completed offline extensions

The following cases use separate catalogues. Captured inputs stay intact. The
optional live acquisition is not run; the source script and command remain in
`README.md`.

In [ ]:
coordinator = ensure_case('interrupted-coordinator', 'cases/coordinator', expected=(87,))
coordinator_before = capture('coordinator_before', coordinator)
assert len(coordinator_before['attempts']) == 1
assert coordinator_before['summary']['counts']['complete'] == 1
assert coordinator_before['summary']['counts']['unobserved'] == 5
saved_report(coordinator, 'coordinator_before')
attempts_before = query(coordinator, 'SELECT COUNT(*) AS n FROM processing_attempts')[0]['n']
coord_result = cli('run', '--manifest', coordinator)
assert coord_result['complete']
assert len(coord_result['invocations']) == 6 - attempts_before
coordinator_after = capture('coordinator_after', coordinator)
assert len(coordinator_after['attempts']) == 6
saved_report(coordinator, 'coordinator_after')
print('After one committed result, restart launches five jobs; later reruns launch zero.')

The coordinator stopped after one result. Restarting the same manifest reused it and launched the remaining five jobs.


In [ ]:
output_case = ensure_case('missing-output', 'cases/output', expected=(0,))
output_before = capture('output_before', output_case)
assert len(output_before['attempts']) == 6
assert all(a['status'] == 'succeeded' for a in output_before['attempts'])
assert output_before['summary']['counts']['ready'] == 1
assert output_before['summary']['counts']['complete'] == 5
assert len(output_before['unavailable_products']) == 1
saved_report(output_case, 'output_before')
output_result = cli('run', '--manifest', output_case)
assert output_result['complete']
output_after = capture('output_after', output_case)
assert len(output_after['attempts']) == 7
after_by_id = {a['attempt_id']: a for a in output_after['attempts']}
assert all(after_by_id[a['attempt_id']]['status'] == 'succeeded' for a in output_before['attempts'])
saved_report(output_case, 'output_after')
print('Old succeeded product with no available location:', investigate(output_case, 3))

The old attempt stayed succeeded even though its output went missing. A new attempt produced a replacement, and the old attempt and missing location remain in the history.


In [ ]:
import shutil
import batch_store, job_adapter

copies = ROOT / 'cases/copies/batch.json'
if not copies.exists():
    cli('setup', '--directory', copies.parent)
    cli('run', '--manifest', copies)
copy_note_path = copies.parent / 'copy_note.json'
copy_manifest = batch_plan.load(copies)
if not copy_note_path.exists():
    member = copy_manifest['frozen']['members'][0]
    with batch_store.connect(copy_manifest['_database']) as c:
        evidence, _, _ = job_adapter.result_evidence(c, member)
        original = Path(evidence['path'])
        secondary = original.parent / 'secondary-copy.tif'
        shutil.copy2(original, secondary)
        assert processing_store.file_hash(secondary) == evidence['sha256']
        with c:
            c.execute('INSERT INTO processing_artifact_locations VALUES (?,?,?,?,?)',
                      (processing_store.digest([evidence['artifact_id'], str(secondary)]),
                       evidence['artifact_id'], str(secondary), 'available', processing_store.now()))
    held = original.with_suffix('.held.tif')
    original.rename(held)
    copy_note_path.write_text(json.dumps({'artifact_id': evidence['artifact_id'], 'original': str(original),
                                         'secondary': str(secondary), 'held': str(held)}, indent=2))
cli('refresh', '--manifest', copies)
copies_evidence = capture('copies_after_refresh', copies)
assert copies_evidence['summary']['counts']['complete'] == 6
assert len(copies_evidence['attempts']) == 6
assert not investigate(copies, 3)
print('A query using any missing copy returns:', query(copies, """SELECT DISTINCT artifact_id
FROM processing_artifact_locations WHERE status='missing'"""))
print('Q3 using no available copy returns:', investigate(copies, 3))
assert query(copies, "SELECT COUNT(*) AS n FROM processing_artifact_locations WHERE status='missing'")[0]['n'] == 1
saved_report(copies, 'copies_after_refresh')

One copy was missing, but the second still passed verification. Checking for any missing location would give the wrong answer; Q3 looks for products with no available location.


In [ ]:
import yaml

identity_dir = ROOT / 'cases/identity'
identity_dir.mkdir(exist_ok=True)
main_manifest = batch_plan.load(MANIFEST)
original_member = main_manifest['frozen']['members'][0]
original_config = ROOT / 'work/config' / (original_member['member_id'] + '.yml')
storage_config = yaml.safe_load(original_config.read_text())
storage_config['runtime']['output_dir'] = str(identity_dir / 'relocated_outputs')
storage_path = identity_dir / 'storage.yml'
storage_path.write_text(yaml.safe_dump(storage_config, sort_keys=False))
storage_batch = identity_dir / 'storage.json'
if not storage_batch.exists():
    cli('freeze', '--config', storage_path, '--out', storage_batch, '--workspace', identity_dir / 'storage_work')
storage_member = batch_plan.load(storage_batch)['frozen']['members'][0]
assert storage_member['request_id'] == original_member['request_id']
assert cli('plan', '--manifest', storage_batch)['members'][0]['state'] == 'complete'

science_config = yaml.safe_load(storage_path.read_text())
science_config['science']['min_valid_fraction'] = 0.2
science_path = identity_dir / 'science.yml'
science_path.write_text(yaml.safe_dump(science_config, sort_keys=False))
science_batch = identity_dir / 'science.json'
if not science_batch.exists():
    cli('freeze', '--config', science_path, '--out', science_batch, '--workspace', identity_dir / 'science_work')
science_member = batch_plan.load(science_batch)['frozen']['members'][0]
assert science_member['request_id'] != original_member['request_id']
assert cli('plan', '--manifest', science_batch)['members'][0]['state'] == 'ready'
identity_evidence = {'original_request': original_member['request_id'],
                     'storage_request': storage_member['request_id'],
                     'changed_science_request': science_member['request_id'],
                     'changed_setting': {'min_valid_fraction': {'before': 0.1, 'after': 0.2}},
                     'changed_request_executed': False}
(EVIDENCE / 'identity.json').write_text(json.dumps(identity_evidence, indent=2))
print(identity_evidence)

for manifest_path in (MANIFEST, missing, worker, coordinator, output_case, copies):
    assert not query(manifest_path, 'PRAGMA foreign_key_check')
    assert query(manifest_path, 'PRAGMA integrity_check')[0]['integrity_check'] == 'ok'
print('All six exercise catalogues pass integrity and foreign-key checks.')

Changing the output directory kept the same request ID. Changing `min_valid_fraction` from 0.1 to 0.2 created a new request and manifest. I planned that request but did not run it.


## Submit

Keep your notebook, three table definitions, three repairs, query file, one saved report, and the decision note. Include the failed/interrupted evidence as well as the recovered outcome.

The extensions are optional.